In [2]:

import numpy as np
import tensorflow as tf
import csv
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
from scipy.optimize import minimize

In [3]:
data = []

with open('new_training_data.csv', newline='') as csvfile:
    reader = csv.reader(csvfile)
    next(reader)
    for row in reader:
        float_row = [float(item) for item in row[1:]]
        data.append(float_row)

data = np.array(data)
print(data[0,0:6],data[0,7])


[4.e-01 2.e+02 5.e+02 5.e-01 1.e-01 4.e-01] 98.75312743529798


In [4]:
param_bounds = [
    (0, 1),      # IR
    (50, 500),   # NG
    (50, 500),   # PS
    (0, 1),      # PC
    (0, 1),      # PM
    (0.4,0.7)      # r1
]

def normalize_params(X, bounds):
    X_norm = np.empty_like(X)
    for i, (min_val, max_val) in enumerate(bounds):
        X_norm[:, i] = (X[:, i] - min_val) / (max_val - min_val)
    return X_norm

def denormalize_params(X_norm, bounds):
    X = np.empty_like(X_norm)
    for i, (min_val, max_val) in enumerate(bounds):
        X[:, i] = X_norm[:, i] * (max_val - min_val) + min_val
    return X

In [5]:

X = data[:, 0:6]  # Hyperparameters
y = data[:, 7:]  # Results

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
X_normalized = normalize_params(X, param_bounds)
print(X_normalized)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

model = keras.Sequential([
    layers.Input(shape=(6,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')

[[0.4        0.33333333 1.         0.5        0.1        0.        ]
 [0.4        0.33333333 1.         0.5        0.1        0.        ]
 [0.4        0.33333333 1.         0.5        0.1        0.        ]
 ...
 [0.9        0.33333333 1.         0.8        0.3        1.        ]
 [0.9        0.33333333 1.         0.8        0.3        1.        ]
 [0.9        0.33333333 1.         0.8        0.3        1.        ]]


In [6]:
model.fit(X_normalized, y_scaled, epochs=500,validation_split=0.2, verbose=0)

In [8]:
# def objective(params_original_scale):
#     params = np.array(params_original_scale).reshape(1, -1)
#     params_scaled = scaler_X.transform(params)
#     pred_scaled = model.predict(params_scaled, verbose=0)
#     return -pred_scaled[0, 0]  # Negative because we want to maximize
def postprocess_params(params):
    params = params.flatten()
    
    # PS: index 2 - round to nearest even integer in bounds
    ps = int(round(params[2]))
    if ps < 50:
        ps = 50
    elif ps > 500:
        ps = 500
    # make it even
    if ps % 2 != 0:
        ps += 1 if ps < 500 else -1
    
    # NMP: index 5 - round to integer within bounds
    nmp = params[5]
    nmp = max(0.4, min(nmp, 0.7))
    
    # NG: index 1 - if integer needed
    ng = int(round(params[1]))
    ng = max(50, min(ng, 500))
    
    # IR, PC, PM remain as floats, just clip between 0 and 1
    ir = np.clip(params[0], 0, 1)
    pc = np.clip(params[3], 0, 1)
    pm = np.clip(params[4], 0, 1)
    
    return np.array([ir, ng, ps, pc, pm, nmp])

def objective(normalized_params):
    normalized_params = np.array(normalized_params).reshape(1, -1)  # shape (1,6)
    pred_scaled = model.predict(normalized_params, verbose=0)       # model expects normalized inputs
    return -pred_scaled[0, 0]  # negate to maximize

# Initial guess: average of input parameters, normalized
initial_guess = normalize_params(np.mean(X, axis=0).reshape(1, -1), param_bounds)[0]
print("Initial guess (normalized):", initial_guess)

bounds = [(0, 1)] * len(param_bounds)

# Run optimization in normalized space
result = minimize(objective, initial_guess, bounds=bounds, method='L-BFGS-B')
print("Optimization result:", result)
best_params_normalized = result.x.reshape(1, -1)

# Denormalize best params back to original scale
best_params = denormalize_params(best_params_normalized, param_bounds)

# Predict the best result using normalized inputs (no scaler_X!)
best_result_scaled = model.predict(best_params_normalized, verbose=0)

# Inverse transform output scaling to get final result in original scale
best_result = scaler_y.inverse_transform(best_result_scaled)

# Optional postprocessing if you have a function to adjust best_params (rounding, etc.)
best_params = postprocess_params(best_params)



print("✅ Best parameters found (original scale):", best_params)
print("🎯 Predicted result with these params:", best_result[0, 0])

Initial guess (normalized): [0.63333333 0.33333333 1.         0.65       0.18333333 0.55555556]
Optimization result:   message: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
  success: True
   status: 0
      fun: -0.31208223
        x: [ 6.333e-01  3.333e-01  1.000e+00  1.000e+00  1.833e-01
             5.556e-01]
      nit: 1
      jac: [ 0.000e+00  0.000e+00 -0.000e+00 -0.000e+00  0.000e+00
             0.000e+00]
     nfev: 14
     njev: 2
 hess_inv: <6x6 LbfgsInvHessProduct with dtype=float64>
✅ Best parameters found (original scale): [6.33333333e-01 2.00000000e+02 5.00000000e+02 1.00000000e+00
 1.83333333e-01 5.66666667e-01]
🎯 Predicted result with these params: 98.5764


In [ ]:
import numpy as np
import subprocess
import csv
import jpype
import jpype.imports
from scipy.optimize import minimize

# ==== Helper Functions ====

def postprocess_params(params):
    # Ensure it's a clean copy and 1D
    params = params.flatten().copy()

    # Process each parameter individually
    ps = int(round(params[2]))
    ps = max(50, min(ps, 500))
    if ps % 2 != 0:
        ps += 1 if ps < 500 else -1

    nmp = int(round(params[5]))
    nmp = max(3, min(nmp, 15))

    ng = int(round(params[1]))
    ng = max(50, min(ng, 500))

    ir = float(np.clip(params[0], 0, 1))
    pc = float(np.clip(params[3], 0, 1))
    pm = float(np.clip(params[4], 0, 1))

    numeric_params = np.array([ir, ng, ps, pc, pm, nmp])

    # String version for Java input
    java_params = [
        f"{ir:.6f}",  # float
        str(ng),      # int
        str(ps),      # int
        f"{pc:.6f}",  # float
        f"{pm:.6f}",  # float
        str(nmp)      # int
    ]

    return numeric_params, java_params


def run_java_algorithm(params):
    if not jpype.isJVMStarted():
        jpype.startJVM(classpath=["C:/Users/USER/Desktop/my_projects/optimization_with_java/bin"])
    GGA = jpype.JClass("GGA.GGA")  # Just the class name
    java_params = jpype.JArray(jpype.JString)([str(p) for p in params])
    print("→ Running Java GGA with params:", java_params)
    GGA.main(java_params)


def read_new_training_data(filepath='training_data.csv'):
    new_data = []
    with open(filepath, newline='') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for row in reader:
            float_row = [float(item) for item in row[1:]]
            new_data.append(float_row)
    return np.array(new_data)

def retrain_model(model, scaler_X, scaler_y, new_data):
    X_new = new_data[:, 0:6]
    y_new = new_data[:, 7:]
    X_new_scaled = scaler_X.transform(X_new)
    y_new_scaled = scaler_y.transform(y_new.reshape(-1, 1))
    model.fit(X_new_scaled, y_new_scaled, epochs=50, validation_split=0.2)

def objective(normalized_params):
    params = denormalize_params(np.array(normalized_params).reshape(1, -1), param_bounds)
    params_scaled = scaler_X.transform(params)
    pred_scaled = model.predict(params_scaled, verbose=0)
    return -pred_scaled[0, 0]

def optimize_params():
    initial_guess = normalize_params(np.mean(X, axis=0).reshape(1, -1), param_bounds)[0]
    bounds = [(0, 1)] * 6
    result = minimize(objective, initial_guess, bounds=bounds, method='L-BFGS-B')
    return result.x

# ==== Main Retraining Loop ====
average_result=0
max_average=0
best_possible_params=[]
num_iterations = 5  # Number of retrain cycles
for iteration in range(num_iterations):
        # Step 1: Optimize best parameters based on the current model
    best_params_norm = optimize_params()
    best_params = denormalize_params(best_params_norm.reshape(1, -1), param_bounds)

    # Step 2: Postprocess (Java & numeric)
    numeric_params, java_params = postprocess_params(best_params)
    print("→ Optimized & Postprocessed Params:", java_params)

    # Step 3: Run Java
    run_java_algorithm(java_params)

    # Step 4: Retrain
    new_training_data = read_new_training_data()
    print("→ New training data shape:", new_training_data.shape)
    average_result = np.mean(new_training_data[:, -1])  # assuming result is in the last column
    print(f"→ Average Result in Training Data: {average_result:.4f}")
    if average_result > max_average:
        max_average = average_result
        print(f"→ New Maximum Result Found: {max_average:.4f}")
        best_possible_params = numeric_params
    retrain_model(model, scaler_X, scaler_y, new_training_data)

    # Step 5: After retraining, re-optimize again
    best_params_norm = optimize_params()
    best_params = denormalize_params(best_params_norm.reshape(1, -1), param_bounds)
    numeric_params, java_params = postprocess_params(best_params)

    # Step 6: Predict
    normalized_numeric = normalize_params(numeric_params.reshape(1, -1), param_bounds)
    best_params_scaled = scaler_X.transform(normalized_numeric)
    pred_scaled = model.predict(best_params_scaled)
    pred = scaler_y.inverse_transform(pred_scaled)

    print("→ Generated Parameters After Retraining:")
    print(f"   IR  = {numeric_params[0]:.3f}   NG  = {int(numeric_params[1])} PS  = {int(numeric_params[2])} (even)PC  = {numeric_params[3]:.3f} PM  = {numeric_params[4]:.3f} NMP = {int(numeric_params[5])}")
    print(f"   Predicted Accuracy: {pred[0, 0]:.3f}")

print("Best possible parameters found during iterations:")
print(f"IR  = {best_possible_params[0]:.3f}   NG  = {int(best_possible_params[1])} PS  = {int(best_possible_params[2])} (even)PC  = {best_possible_params[3]:.3f} PM  = {best_possible_params[4]:.3f} NMP = {int(best_possible_params[5])}")